In [1]:
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv(Path.cwd().parent / ".env")

url = (
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)
engine = create_engine(url)

In [2]:
clientes = pd.read_sql("SELECT * FROM clientes", engine)
produtos = pd.read_sql("SELECT * FROM produtos", engine)
pedidos  = pd.read_sql("SELECT * FROM pedidos", engine)
itens    = pd.read_sql("SELECT * FROM itens_pedido", engine)

In [3]:
for nome, df in [("clientes", clientes), ("produtos", produtos),
                 ("pedidos", pedidos), ("itens", itens)]:
    print(f"--- {nome}: {df.shape[0]} linhas, {df.shape[1]} colunas")
    print(df.head(3), "\n")

--- clientes: 500 linhas, 6 colunas
   id_cliente               nome                 email            cidade  \
0           1       Brenda Alves  cliente1@exemplo.com  da Cruz de Goiás   
1           2         Cauã Rocha  cliente2@exemplo.com            Pastor   
2           3  Ana Beatriz Alves  cliente3@exemplo.com             Cunha   

  estado data_cadastro  
0     RJ    2023-01-26  
1     RS    2023-09-08  
2     PR    2023-05-23   

--- produtos: 40 linhas, 4 colunas
   id_produto              nome    categoria  preco
0           1    Fone Bluetooth  Eletrônicos  149.9
1           2  Carregador Turbo  Eletrônicos   79.9
2           3        Smartwatch  Eletrônicos  399.0 

--- pedidos: 3000 linhas, 4 colunas
   id_pedido  id_cliente data_pedido    status
0          1          86  2024-11-02  entregue
1          2          24  2024-11-28  entregue
2          3         151  2025-01-02  entregue 

--- itens: 7562 linhas, 5 colunas
   id_item  id_pedido  id_produto  quantidade  preco

In [4]:
pedidos.info()
print(pedidos.isna().sum())
print(pedidos["status"].value_counts())

<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id_pedido    3000 non-null   int64 
 1   id_cliente   3000 non-null   int64 
 2   data_pedido  3000 non-null   object
 3   status       3000 non-null   str   
dtypes: int64(2), object(1), str(1)
memory usage: 93.9+ KB
id_pedido      0
id_cliente     0
data_pedido    0
status         0
dtype: int64
status
entregue     2567
cancelado     206
enviado       128
pendente       99
Name: count, dtype: int64


In [5]:
itens.describe()

,id_item,id_pedido,id_produto,quantidade,preco_unitario
count,7562.000000,7562.000000,7562.000000,7562.000000,7562.000000
mean,3781.500000,1495.881777,20.357048,1.981354,131.430217
std,2183.105701,867.275720,11.641847,0.822766,146.375081
min,1.000000,1.000000,1.000000,1.000000,19.900000
25%,1891.250000,743.000000,10.000000,1.000000,49.900000
50%,3781.500000,1496.500000,20.000000,2.000000,89.900000
75%,5671.750000,2248.000000,30.000000,3.000000,159.900000
max,7562.000000,3000.000000,40.000000,3.000000,899.000000


In [6]:
pedidos["data_pedido"] = pd.to_datetime(pedidos["data_pedido"])
clientes["data_cadastro"] = pd.to_datetime(clientes["data_cadastro"])

itens["valor_item"] = itens["quantidade"] * itens["preco_unitario"]
itens["valor_item"] = itens["valor_item"].astype(float)

In [7]:
base = (
    itens
    .merge(pedidos, on="id_pedido")
    .merge(produtos[["id_produto", "nome", "categoria"]], on="id_produto")
    .merge(clientes[["id_cliente", "nome", "estado"]], on="id_cliente",
           suffixes=("_produto", "_cliente"))
)

vendas = base[base["status"] != "cancelado"].copy()
print(base.shape, vendas.shape)

(7562, 13) (7040, 13)


In [8]:
faturamento = vendas["valor_item"].sum()
n_pedidos = vendas["id_pedido"].nunique()
ticket_medio = faturamento / n_pedidos

print(f"Faturamento total: R$ {faturamento:,.2f}")
print(f"Pedidos: {n_pedidos}")
print(f"Ticket médio: R$ {ticket_medio:,.2f}")

Faturamento total: R$ 1,836,430.20
Pedidos: 2794
Ticket médio: R$ 657.28


In [9]:
por_categoria = (
    vendas.groupby("categoria")["valor_item"].sum()
    .sort_values(ascending=False)
)
print(por_categoria)
print((por_categoria / por_categoria.sum() * 100).round(2))

categoria
Eletrônicos    486358.4
Esportes       436502.2
Casa           406784.1
Moda           342466.1
Livros         164319.4
Name: valor_item, dtype: float64
categoria
Eletrônicos    26.48
Esportes       23.77
Casa           22.15
Moda           18.65
Livros          8.95
Name: valor_item, dtype: float64


In [10]:
por_mes = (
    vendas.groupby(vendas["data_pedido"].dt.to_period("M"))
    .agg(faturamento=("valor_item", "sum"),
         pedidos=("id_pedido", "nunique"))
)
print(por_mes)

             faturamento  pedidos
data_pedido                      
2024-01          63488.9       97
2024-02          76159.4      124
2024-03          91397.4      138
2024-04          90642.6      135
2024-05          95123.4      141
2024-06          77929.5      121
2024-07          79889.9      112
2024-08          58588.7      101
2024-09          75317.4      118
2024-10          81725.2      130
2024-11          82307.3      141
2024-12          88567.3      131
2025-01          96896.9      148
2025-02          85863.7      125
2025-03          80288.0      139
2025-04          82293.0      122
2025-05          92610.2      136
2025-06          87283.2      129
2025-07          86926.6      130
2025-08         109171.3      152
2025-09          79894.0      119
2025-10          74066.3      105


In [11]:
top_produtos = (
    vendas.groupby(["nome_produto", "categoria"])["quantidade"].sum()
    .sort_values(ascending=False)
    .head(10)
)
print(top_produtos)

nome_produto        categoria  
Tapete de Yoga      Esportes       417
Jogo de Panelas     Casa           398
Mochila             Moda           386
Aspirador Portátil  Casa           385
Bola de Futebol     Esportes       379
Guia de Dados       Livros         377
Luminária LED       Casa           377
Organizador         Casa           371
Carregador Turbo    Eletrônicos    371
HQ Clássica         Livros         370
Name: quantidade, dtype: int64


In [12]:
valor_pedido = vendas.groupby(["id_pedido", "id_cliente", "nome_cliente"],
                              as_index=False)["valor_item"].sum()

top_clientes = (
    valor_pedido.groupby(["id_cliente", "nome_cliente"])
    .agg(pedidos=("id_pedido", "count"),
         total_gasto=("valor_item", "sum"),
         ticket_medio=("valor_item", "mean"))
    .sort_values("total_gasto", ascending=False)
    .head(10)
    .round(2)
)
print(top_clientes)

                                  pedidos  total_gasto  ticket_medio
id_cliente nome_cliente                                             
86         Pedro Henrique Abreu        10      11068.8       1106.88
344        Pedro Lucas Andrade         10      10903.3       1090.33
388        Gael Henrique Pereira        9      10655.5       1183.94
189        Clarice Monteiro             7       9888.0       1412.57
68         Rebeca da Cunha              9       9616.1       1068.46
262        João Gabriel Macedo         10       9232.0        923.20
199        Maria Sophia Nunes           9       9119.2       1013.24
21         Dra. Zoe da Luz              6       8745.1       1457.52
329        Dr. Erick Vasconcelos        9       8566.0        951.78
113        Dr. Mathias Leão             8       8510.0       1063.75
